In [4]:
import os
import yt_dlp
from faster_whisper import WhisperModel


def download_audio(youtube_url, output_dir="downloads"):
    os.makedirs(output_dir, exist_ok=True)

    ydl_opts = {
        "format": "bestaudio/best",
        "outtmpl": f"{output_dir}/%(title)s.%(ext)s",
        "postprocessors": [
            {
                "key": "FFmpegExtractAudio",
                "preferredcodec": "mp3",
                "preferredquality": "192",
            }
        ],
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(youtube_url, download=True)
        title = info["title"]

    audio_path = os.path.join(output_dir, f"{title}.mp3")
    return audio_path, title


def transcribe_audio(audio_path, output_txt):
    model = WhisperModel("base", device="cpu", compute_type="int8")

    segments, info = model.transcribe(audio_path)

    with open(output_txt, "w", encoding="utf-8") as f:
        f.write(f"Detected language: {info.language}\n\n")

        for segment in segments:
            line = f"[{segment.start:.2f} - {segment.end:.2f}] {segment.text}\n"
            f.write(line)

    print(f"Saved transcript to: {output_txt}")


if __name__ == "__main__":
    url = input("Paste YouTube URL: ")

    output_dir = "downloads"
    audio_path, title = download_audio(url, output_dir=output_dir)
    output_txt = os.path.join(output_dir, f"{title}_transcript.txt")

    transcribe_audio(audio_path, output_txt)

[youtube] Extracting URL: https://www.youtube.com/live/0qnWvTKTrJ0
[youtube] 0qnWvTKTrJ0: Downloading webpage


[youtube] 0qnWvTKTrJ0: Downloading android vr player API JSON
[info] 0qnWvTKTrJ0: Downloading 1 format(s): 251
[download] Destination: downloads/Live Tira-Dúvidas GovTech Economia.webm
[download] 100% of   56.95MiB in 00:00:27 at 2.10MiB/s     
[ExtractAudio] Destination: downloads/Live Tira-Dúvidas GovTech Economia.mp3
Deleting original file downloads/Live Tira-Dúvidas GovTech Economia.webm (pass -k to keep)


config.json: 0.00B [00:00, ?B/s]

vocabulary.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.bin:   0%|          | 0.00/145M [00:00<?, ?B/s]

Saved transcript to: Live Tira-Dúvidas GovTech Economia_transcript.txt
